In [ ]:
from google.colab import files
uploaded = files.upload()

Saving retail_transactions_100k.csv to retail_transactions_100k.csv


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("RetailDataAnalysis") \
    .getOrCreate()

In [ ]:
df = spark.read.csv(
    "retail_transactions_100k.csv",
    header=True,
    inferSchema=True
)

In [ ]:
df.show(5)

+--------+----------+-----------+--------------+---------+------+----------+---------------+-----------+--------+----------+------------+--------------+---------+------+
|order_id|order_date|customer_id| customer_name|     city|region|product_id|   product_name|   category|quantity|unit_price|discount_pct|payment_method|   status|rating|
+--------+----------+-----------+--------------+---------+------+----------+---------------+-----------+--------+----------+------------+--------------+---------+------+
| O000001|2026-02-27|     C00205|Customer_00205|     Pune|  West|      P004|27 Inch Monitor|Electronics|       2|  16961.87|           0|           COD|Delivered|  NULL|
| O000002|2026-04-22|     C01906|Customer_01906|  Lucknow| North|      P010|     Web Camera|Accessories|       1|   4241.16|          20|   Net Banking|Delivered|     1|
| O000003|2026-03-23|     C03463|Customer_03463|Ahmedabad|  West|      P005|   Office Chair|  Furniture|       2|   8590.05|          10|           UP

In [ ]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- rating: integer (nullable = true)



In [ ]:
print("Total records:", df.count())

Total records: 100000


In [ ]:
# 1. Display the first 20 records
df.show(20)

# 2. Display the schema
df.printSchema()

# 3. Display all column names
print(df.columns)

# 4. Count the total number of records
df.count()

# 5. Display specific columns (order ID, product name, city, quantity, unit price)
df.select("order_id", "product_name", "city", "quantity", "unit_price").show()

# 6. Display only Electronics transactions
df.filter(df.category == "Electronics").show()

# 7. Display transactions from Hyderabad
df.filter(df.city == "Hyderabad").show()

# 8. Display transactions where quantity is greater than 3
df.filter(df.quantity > 3).show()

# 9. Display transactions where unit price is greater than 20,000
df.filter(df.unit_price > 20000).show()

# 10. Display Delivered transactions
df.filter(df.status == "Delivered").show()

+--------+----------+-----------+--------------+---------+------+----------+-------------------+-----------+--------+----------+------------+--------------+----------+------+
|order_id|order_date|customer_id| customer_name|     city|region|product_id|       product_name|   category|quantity|unit_price|discount_pct|payment_method|    status|rating|
+--------+----------+-----------+--------------+---------+------+----------+-------------------+-----------+--------+----------+------------+--------------+----------+------+
| O000001|2026-02-27|     C00205|Customer_00205|     Pune|  West|      P004|    27 Inch Monitor|Electronics|       2|  16961.87|           0|           COD| Delivered|  NULL|
| O000002|2026-04-22|     C01906|Customer_01906|  Lucknow| North|      P010|         Web Camera|Accessories|       1|   4241.16|          20|   Net Banking| Delivered|     1|
| O000003|2026-03-23|     C03463|Customer_03463|Ahmedabad|  West|      P005|       Office Chair|  Furniture|       2|   8590.

In [ ]:
from pyspark.sql.functions import col

# 11. Find Electronics transactions from Hyderabad
df.filter((col("category") == "Electronics") & (col("city") == "Hyderabad")).show()

# 12. Find Delivered transactions from the South region
df.filter((col("status") == "Delivered") & (col("region") == "South")).show()

# 13. Find orders having quantity greater than 2 and unit price greater than 10,000
df.filter((col("quantity") > 2) & (col("unit_price") > 10000)).show()

# 14. Find transactions from either Mumbai or Pune
df.filter(col("city").isin(["Mumbai", "Pune"])).show()

# 15. Find transactions that are not Cancelled
df.filter(col("status") != "Cancelled").show()

# 16. Find Furniture products costing more than 8,000
df.filter((col("category") == "Furniture") & (col("unit_price") > 8000)).show()

# 17. Find Accessories products with quantity greater than 3
df.filter((col("category") == "Accessories") & (col("quantity") > 3)).show()

+--------+----------+-----------+--------------+---------+------+----------+---------------+-----------+--------+----------+------------+--------------+----------+------+
|order_id|order_date|customer_id| customer_name|     city|region|product_id|   product_name|   category|quantity|unit_price|discount_pct|payment_method|    status|rating|
+--------+----------+-----------+--------------+---------+------+----------+---------------+-----------+--------+----------+------------+--------------+----------+------+
| O000015|2026-06-07|     C01962|Customer_01962|Hyderabad| South|      P004|27 Inch Monitor|Electronics|       5|  19287.21|           0|   Net Banking|Processing|  NULL|
| O000032|2026-08-05|     C02267|Customer_02267|Hyderabad| South|      P001|     Laptop Pro|Electronics|       3|  67819.64|           5|   Credit Card|Processing|  NULL|
| O000063|2026-05-22|     C04554|Customer_04554|Hyderabad| South|      P009|    Smart Watch|Electronics|       2|  11204.38|          10|   Net B

In [ ]:
# 18. Select only order_id, customer_name, product_name, unit_price
df.select("order_id", "customer_name", "product_name", "unit_price").show()

# 19 & 20. Rename columns (unit_price -> price, customer_name -> customer)
df_renamed = df.withColumnRenamed("unit_price", "price") \
               .withColumnRenamed("customer_name", "customer")

# 21 & 22. Drop columns
df_dropped_rating = df.drop("rating")
df_new_dropped = df.drop("customer_name", "rating", "payment_method")

# 23. Add gross_amount (quantity * unit_price)
df = df.withColumn("gross_amount", col("quantity") * col("unit_price"))

# 24. Add discount_amount (gross_amount * discount_pct / 100)
df = df.withColumn("discount_amount", col("gross_amount") * (col("discount_pct") / 100))

# 25. Add final_amount (gross_amount - discount_amount)
df = df.withColumn("final_amount", col("gross_amount") - col("discount_amount"))

# 26. Add constant column country = 'India'
from pyspark.sql.functions import lit
df = df.withColumn("country", lit("India"))

# 27. Add high_value (Yes if final_amount >= 50000 else No)
from pyspark.sql.functions import when
df = df.withColumn("high_value", when(col("final_amount") >= 50000, "Yes").otherwise("No"))

+--------+--------------+-------------------+----------+
|order_id| customer_name|       product_name|unit_price|
+--------+--------------+-------------------+----------+
| O000001|Customer_00205|    27 Inch Monitor|  16961.87|
| O000002|Customer_01906|         Web Camera|   4241.16|
| O000003|Customer_03463|       Office Chair|   8590.05|
| O000004|Customer_00356|        Smart Watch|  12908.38|
| O000005|Customer_01576|         Laptop Pro|  67839.91|
| O000006|Customer_02989|         Study Desk|   13349.3|
| O000007|Customer_01339|             Tablet|  30193.06|
| O000008|Customer_01877|         Study Desk|  13479.74|
| O000009|Customer_03242|Mechanical Keyboard|   4240.53|
| O000010|Customer_01797|        Smart Watch|  11214.55|
| O000011|Customer_03153|         Web Camera|   4219.56|
| O000012|Customer_02186|     Wireless Mouse|   1484.34|
| O000013|Customer_04101|        Smart Watch|   12711.4|
| O000014|Customer_04419|         Laptop Pro|  63171.22|
| O000015|Customer_01962|    27

In [ ]:
# 28. Create order_size (High, Medium, Low)
df = df.withColumn("order_size",
    when(col("final_amount") > 50000, "High")
    .when(col("final_amount") >= 15000, "Medium")
    .otherwise("Low")
)

# 29. Create quantity_type (Bulk if quantity >= 4 else Regular)
df = df.withColumn("quantity_type", when(col("quantity") >= 4, "Bulk").otherwise("Regular"))

# 30. Create delivery-status categories if needed based on your status values
df = df.withColumn("delivery_category",
    when(col("status") == "Delivered", "Completed")
    .when(col("status").isin(["Cancelled", "Returned"]), "Failed")
    .otherwise("In Progress")
)

In [ ]:
# 31. Sort by unit price highest to lowest
df.orderBy(col("unit_price").desc()).show()

# 33. Sort alphabetically by city
df.orderBy(col("city").asc()).show()

# 35. Display 10 highest-value transactions
df.orderBy(col("final_amount").desc()).limit(10).show()

# 36. Display all unique cities[cite: 6]
df.select("city").distinct().show()

# 40. Count unique customers[cite: 6]
df.select("customer_id").distinct().count()

# 42-48. Aggregations
from pyspark.sql.functions import sum, avg, min, max
df.select(
    sum("quantity").alias("total_quantity"),
    sum("gross_amount").alias("total_gross_sales"),
    avg("unit_price").alias("avg_unit_price"),
    max("unit_price").alias("max_unit_price"),
    min("unit_price").alias("min_unit_price"),
    sum("discount_amount").alias("total_discount"),
    sum("final_amount").alias("total_final_sales")
).show()

+--------+----------+-----------+--------------+---------+------+----------+------------+-----------+--------+----------+------------+--------------+----------+------+------------------+------------------+------------------+-------+----------+----------+-------------+-----------------+
|order_id|order_date|customer_id| customer_name|     city|region|product_id|product_name|   category|quantity|unit_price|discount_pct|payment_method|    status|rating|      gross_amount|   discount_amount|      final_amount|country|high_value|order_size|quantity_type|delivery_category|
+--------+----------+-----------+--------------+---------+------+----------+------------+-----------+--------+----------+------------+--------------+----------+------+------------------+------------------+------------------+-------+----------+----------+-------------+-----------------+
| O016427|2026-01-20|     C01275|Customer_01275|Ahmedabad|  West|      P001|  Laptop Pro|Electronics|       1|  70199.47|           0|     

In [ ]:
# 49. Count transactions by status
df.groupBy("status").count().show()

# 52. Calculate total sales by region
df.groupBy("region").sum("final_amount").show()

# 62. Find the top 5 cities based on total sales
df.groupBy("city") \
  .sum("final_amount") \
  .withColumnRenamed("sum(final_amount)", "total_sales") \
  .orderBy(col("total_sales").desc()) \
  .limit(5).show()

# 65-67. Date processing (extract year, month, day)
from pyspark.sql.functions import year, month, dayofmonth
df = df.withColumn("year", year("order_date")) \
       .withColumn("month", month("order_date")) \
       .withColumn("day", dayofmonth("order_date"))

# 73-76. Check for missing values
df.filter(col("payment_method").isNull()).show()
print("Missing payment methods count:", df.filter(col("payment_method").isNull()).count())

# 77 & 78. Replace missing values
df = df.na.fill({"payment_method": "Unknown", "rating": 0})

# 79. Drop records having missing payment methods
df = df.dropna(subset=["payment_method"])

+----------+-----+
|    status|count|
+----------+-----+
|  Returned| 5912|
|   Shipped|13950|
|Processing|12115|
| Cancelled| 9927|
| Delivered|58096|
+----------+-----+

+------+--------------------+
|region|   sum(final_amount)|
+------+--------------------+
| South|1.2736728779755006E9|
|  East| 4.188441210040009E8|
|  West|1.2745074508245018E9|
| North|1.2742773118394942E9|
+------+--------------------+

+---------+--------------------+
|     city|         total_sales|
+---------+--------------------+
|     Pune| 4.380059195804999E8|
|Ahmedabad|4.2995853770049965E8|
|Hyderabad| 4.282570748114998E8|
|  Lucknow|4.2674751151249886E8|
|Bangalore|4.2477450649949884E8|
+---------+--------------------+

+--------+----------+-----------+--------------+---------+------+----------+-------------------+-----------+--------+----------+------------+--------------+----------+------+------------------+------------------+------------------+-------+----------+----------+-------------+--------------

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Widgets") \
    .getOrCreate()

In [ ]:
from google.colab import files,output
output.enable_custom_widget_manager()
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
uploaded=files.upload()
df=pd.read_csv('manufacturing_production_extra_12k.csv')
print("Dataset loaded successfully!")
print("Number of records:", len(df))
display(df.head())


Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k (2).csv
Dataset loaded successfully!
Number of records: 12500


,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0120001,2026-06-01,PL05,MC024,OP0191,Evening,PMP-210,Pumps,293,292,10,44.0,120.42,70.59,2.21,Pass,Yes,Anita
1,PR0120002,2026-09-22,PL01,MC001,OP0461,Night,MTR-100,Motors,1009,897,25,59.0,317.23,81.74,3.03,Pass,No,Anita
2,PR0120003,2026-05-24,PL03,MC012,OP0451,Evening,MTR-100,Motors,960,983,34,0.0,350.82,67.29,2.01,Pass,No,Deepa
3,PR0120004,2026-09-12,PL04,MC016,OP0237,Night,CTL-130,Controllers,310,320,15,14.0,142.34,59.70,3.48,Pass,No,Anita
4,PR0120005,2026-02-04,PL04,MC018,OP0072,Morning,VLV-220,Valves,957,979,22,14.0,296.70,62.52,2.33,Pass,No,Harini


In [ ]:
# Function to create dropdown options
def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [
        (value, value) for value in values
    ]


# 1. Plant dropdown
plant_dropdown = widgets.Dropdown(
    options=create_options("plant_id", "All Plants"),
    description="Plant:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 2. Product family dropdown
product_dropdown = widgets.Dropdown(
    options=create_options("product_family", "All Products"),
    description="Product Family:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)


# 3. Quality status dropdown
quality_dropdown = widgets.Dropdown(
    options=create_options("quality_status", "All Statuses"),
    description="Quality:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 4. Slider for units produced
units = pd.to_numeric(df["units_produced"], errors="coerce").dropna()

units_slider = widgets.IntRangeSlider(
    value=(int(units.min()), int(units.max())),
    min=int(units.min()),
    max=int(units.max()),
    step=1,
    description="Units Produced:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)

In [ ]:
from IPython.display import display
from google.colab import output

# Enable interactive widgets in Google Colab
output.enable_custom_widget_manager()

# Display all four widgets
display(widgets.VBox([
    plant_dropdown,
    product_dropdown,
    quality_dropdown,
    units_slider
]))

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from google.colab import files, output

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Number of records:", len(df))
display(df.head())

Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k (10).csv
Dataset loaded successfully!
Number of records: 12500


,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0120001,2026-06-01,PL05,MC024,OP0191,Evening,PMP-210,Pumps,293,292,10,44.0,120.42,70.59,2.21,Pass,Yes,Anita
1,PR0120002,2026-09-22,PL01,MC001,OP0461,Night,MTR-100,Motors,1009,897,25,59.0,317.23,81.74,3.03,Pass,No,Anita
2,PR0120003,2026-05-24,PL03,MC012,OP0451,Evening,MTR-100,Motors,960,983,34,0.0,350.82,67.29,2.01,Pass,No,Deepa
3,PR0120004,2026-09-12,PL04,MC016,OP0237,Night,CTL-130,Controllers,310,320,15,14.0,142.34,59.70,3.48,Pass,No,Anita
4,PR0120005,2026-02-04,PL04,MC018,OP0072,Morning,VLV-220,Valves,957,979,22,14.0,296.70,62.52,2.33,Pass,No,Harini


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, desc

spark = SparkSession.builder \
    .appName("LazyExecutionDemo") \
    .getOrCreate()

data = [
    (1, "Hyderabad", "Laptop", 2, 55000),
    (2, "Bangalore", "Mobile", 5, 20000),
    (3, "Chennai", "Tablet", 3, 15000),
    (4, "Hyderabad", "Mobile", 4, 22000),
    (5, "Mumbai", "Laptop", 1, 60000),
    (6, "Bangalore", "Tablet", 6, 12000),
    (7, "Chennai", "Laptop", 2, 58000),
    (8, "Mumbai", "Mobile", 8, 18000),
    (9, "Hyderabad", "Tablet", 5, 14000),
    (10, "Bangalore", "Laptop", 3, 52000),
    (11, "Mumbai", "Tablet", 7, 13000),
    (12, "Chennai", "Mobile", 6, 21000)
]

columns = ["sale_id", "city", "product", "quantity", "price"]

df = spark.createDataFrame(data, columns)

df.show()

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      2|Bangalore| Mobile|       5|20000|
|      3|  Chennai| Tablet|       3|15000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      6|Bangalore| Tablet|       6|12000|
|      7|  Chennai| Laptop|       2|58000|
|      8|   Mumbai| Mobile|       8|18000|
|      9|Hyderabad| Tablet|       5|14000|
|     10|Bangalore| Laptop|       3|52000|
|     11|   Mumbai| Tablet|       7|13000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+



In [ ]:
filtered_df=df.filter(col("quantity")>3)
print("Transformation created")
selected_df=filtered_df.select(
    "city","product","quantity"
)
print("Another transformation created")
selected_df.show()

In [ ]:
step1 = df.filter(col("quantity")>=3)
step2=step1.withColumn(
    "revenue",
    col("quantity") * col("price")
)
step3=step2.groupBy("city").agg(
    sum("revenue").alias("total_revenue")
)
step4=step3.orderBy(desc("total_revenue"))
print("All transformations defined")
step4.show()

In [ ]:
result=df.filter(col("price") > 20000) \
                 .select("city", "product", "price") \
                 .orderBy(desc("price"))
result.explain()

In [ ]:
result.show()

+---------+-------+-----+
|     city|product|price|
+---------+-------+-----+
|   Mumbai| Laptop|60000|
|  Chennai| Laptop|58000|
|Hyderabad| Laptop|55000|
|Bangalore| Laptop|52000|
|Hyderabad| Mobile|22000|
|  Chennai| Mobile|21000|
+---------+-------+-----+



In [ ]:
high_value_sales=df.filter(col("price")>20000)
high_value_sales.show()
print("Total records:", high_value_sales.count())
records=high_value_sales.collect()
print(records)

In [ ]:
from google.colab import files
uploaded = files.upload()
from google.colab import files
uploaded = files.upload()
from google.colab import files
uploaded = files.upload()

Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv


Saving lazy_execution_products.csv to lazy_execution_products.csv


Saving lazy_execution_stores.csv to lazy_execution_stores.csv


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, desc
spark = SparkSession.builder \
    .appName("LazyExecutionExercises") \
    .getOrCreate()
sales_df = spark.read.csv("lazy_execution_sales_25000.csv", header=True, inferSchema=True)
products_df = spark.read.csv("lazy_execution_products.csv", header=True, inferSchema=True)
stores_df = spark.read.csv("lazy_execution_stores.csv", header=True, inferSchema=True)

In [ ]:
filtered_sales = sales_df.filter(col("quantity") > 5)

In [ ]:
selected_sales = sales_df.select("order_id", "product_id", "quantity", "unit_price")
selected_sales.show(10)

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000001|      P002|       3|      2999|
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000005|      P007|       2|     79999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000009|      P007|       5|     79999|
| O000010|      P006|       5|     55999|
+--------+----------+--------+----------+
only showing top 10 rows


In [ ]:
high_price_count = sales_df.filter(col("unit_price") > 20000).count()
print(f"Number of records with unit price > 20,000: {high_price_count}")

Number of records with unit price > 20,000: 6560


In [ ]:
gross_sales = sales_df.withColumn("gross_amount", col("quantity") * col("unit_price"))
gross_sales.explain()
gross_sales.select("order_id", "quantity", "unit_price", "gross_amount").show(10)

== Physical Plan ==
*(1) Project [order_id#128, order_date#129, store_id#130, product_id#131, quantity#132, unit_price#133, discount_pct#134, payment_method#135, order_status#136, sales_channel#137, (quantity#132 * unit_price#133) AS gross_amount#214]
+- FileScan csv [order_id#128,order_date#129,store_id#130,product_id#131,quantity#132,unit_price#133,discount_pct#134,payment_method#135,order_status#136,sales_channel#137] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+--------+----------+------------+
|order_id|quantity|unit_price|gross_amount|
+--------+--------+----------+------------+
| O000001|       3|      2999|        8997|
| O000002|       6|     29999|      179994|
| O000003|       6|     22999|      137994|
| O000004|       7|     2

In [ ]:
completed_sorted = sales_df.filter(col("order_status") == "Completed") \
                           .orderBy(col("unit_price").desc())

top_10_records = completed_sorted.take(10)
for record in top_10_records:
    print(record)

Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online')
Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online')
Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online')
Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online')
Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9, unit_pr

In [ ]:
sales_df.filter(col("order_status") == "Completed") \
        .withColumn("gross_amount", col("quantity") * col("unit_price")) \
        .select("order_id", "store_id", "product_id", "quantity", "gross_amount") \
        .orderBy(col("gross_amount").desc()) \
        .show(10)

+--------+--------+----------+--------+------------+
|order_id|store_id|product_id|quantity|gross_amount|
+--------+--------+----------+--------+------------+
| O001170|     S01|      P007|       9|      719991|
| O002603|     S01|      P007|       9|      719991|
| O001201|     S05|      P007|       9|      719991|
| O000107|     S05|      P007|       9|      719991|
| O001209|     S02|      P007|       9|      719991|
| O000456|     S04|      P007|       9|      719991|
| O001300|     S03|      P007|       9|      719991|
| O000615|     S08|      P007|       9|      719991|
| O001595|     S05|      P007|       9|      719991|
| O000737|     S08|      P007|       9|      719991|
+--------+--------+----------+--------+------------+
only showing top 10 rows


In [ ]:
sales_df.filter(col("discount_pct") >= 10) \
        .withColumn("discounted_sales_amount", (col("quantity") * col("unit_price")) * (1 - col("discount_pct") / 100)) \
        .orderBy(col("discounted_sales_amount").desc()) \
        .show(15)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_sales_amount|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------------+
| O006653|2026-06-29|     S01|      P007|       9|     79999|          10|          Card|   Cancelled|       Online|               647991.9|
| O011893|2026-03-26|     S07|      P007|       9|     79999|          10|           UPI|   Completed|       Online|               647991.9|
| O006699|2026-06-25|     S05|      P007|       9|     79999|          10|   Net Banking|   Completed|       Online|               647991.9|
| O003108|2026-02-23|     S08|      P007|       9|     79999|          10|   Net Banking|   Completed|     In Store|               647991.9|
| O007743|202

In [ ]:
sales_df.groupBy("payment_method") \
        .agg(sum("quantity").alias("total_quantity")) \
        .orderBy(col("total_quantity").desc()) \
        .show()

+--------------+--------------+
|payment_method|total_quantity|
+--------------+--------------+
|          Cash|         31754|
|          Card|         31327|
|           UPI|         30894|
|   Net Banking|         30889|
+--------------+--------------+



In [ ]:
sales_df.groupBy("sales_channel", "order_status") \
        .agg(count("*").alias("transaction_count")) \
        .show()

+-------------+------------+-----------------+
|sales_channel|order_status|transaction_count|
+-------------+------------+-----------------+
|       Online|   Completed|            10797|
|     In Store|   Cancelled|              592|
|       Online|    Returned|              987|
|       Online|   Cancelled|              633|
|     In Store|    Returned|             1013|
|     In Store|   Completed|            10978|
+-------------+------------+-----------------+



In [ ]:
subset_df = sales_df.select("order_id", "store_id", "product_id", "quantity") \
                    .drop("store_id")

first_record = subset_df.first()
print(first_record)

Row(order_id='O000001', product_id='P002', quantity=3)


In [ ]:
sales_df.join(products_df, "product_id", "inner") \
        .select("product_name", "category", "quantity", "unit_price") \
        .show(20)

+-------------------+-----------+--------+----------+
|       product_name|   category|quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|       3|      2999|
|      Smartphone 5G|    Mobiles|       6|     29999|
|     Android Tablet|    Tablets|       6|     22999|
|     Android Tablet|    Tablets|       7|     22999|
|          Laptop 16|  Computers|       2|     79999|
|          Webcam HD|Accessories|       8|      2499|
|  Bluetooth Speaker|      Audio|       8|      3999|
|Mechanical Keyboard|Accessories|       9|      2999|
|          Laptop 16|  Computers|       5|     79999|
|          Laptop 14|  Computers|       5|     55999|
|          Laptop 16|  Computers|       1|     79999|
|  Bluetooth Speaker|      Audio|       9|      3999|
|   External HDD 2TB|    Storage|       6|      5999|
|          Webcam HD|Accessories|       8|      2499|
|          Laptop 16|  Computers|       7|     79999|
|          Laptop 14|  Compu

In [ ]:
sales_df.join(stores_df, "store_id", "inner") \
        .groupBy("city") \
        .agg(sum("quantity").alias("total_quantity_sold")) \
        .show()

+---------+-------------------+
|     city|total_quantity_sold|
+---------+-------------------+
|  Chennai|              15727|
|   Mumbai|              15489|
|  Kolkata|              15619|
|     Pune|              15407|
|    Delhi|              15101|
|Bengaluru|              15718|
|Hyderabad|              15784|
|   Jaipur|              16019|
+---------+-------------------+



In [ ]:
sales_df.join(products_df, "product_id", "inner") \
        .join(stores_df, "store_id", "inner") \
        .filter(col("order_status") == "Completed") \
        .withColumn("revenue", col("quantity") * col("unit_price")) \
        .groupBy("category", "region") \
        .agg(sum("revenue").alias("total_revenue")) \
        .orderBy("category", "region") \
        .show(20)

+-----------+------+-------------+
|   category|region|total_revenue|
+-----------+------+-------------+
|Accessories|  East|      9093361|
|Accessories| North|     18463455|
|Accessories| South|     28139018|
|Accessories|  West|     18351560|
|      Audio|  East|     12043135|
|      Audio| North|     24327211|
|      Audio| South|     34629665|
|      Audio|  West|     25317111|
|  Computers|  East|    122990192|
|  Computers| North|    240756425|
|  Computers| South|    371954527|
|  Computers|  West|    221092729|
|   Displays|  East|     13616199|
|   Displays| North|     32468090|
|   Displays| South|     45761308|
|   Displays|  West|     28286336|
|    Mobiles|  East|     41178126|
|    Mobiles| North|     75482543|
|    Mobiles| South|    120674524|
|    Mobiles|  West|     80372388|
+-----------+------+-------------+
only showing top 20 rows


In [ ]:
sales_df.filter(col("order_status") == "Completed") \
        .join(products_df, "product_id", "inner") \
        .withColumn("discounted_revenue", (col("quantity") * col("unit_price")) * (1 - col("discount_pct") / 100)) \
        .groupBy("product_name") \
        .agg(sum("discounted_revenue").alias("total_discounted_revenue")) \
        .orderBy(col("total_discounted_revenue").desc()) \
        .show(5)

+---------------+------------------------+
|   product_name|total_discounted_revenue|
+---------------+------------------------+
|      Laptop 16|    5.2272546585000014E8|
|      Laptop 14|     3.830051604999999E8|
|  Smartphone 5G|     2.060406317500002E8|
| Android Tablet|    1.5040656029999995E8|
|27-inch Monitor|    1.1406074015000013E8|
+---------------+------------------------+
only showing top 5 rows


In [ ]:
t1 = sales_df.filter(col("order_status") == "Completed")
print("Transformation 1 applied (Filter). No execution yet.")

t2 = t1.select("order_id", "store_id", "product_id", "quantity", "unit_price")
print("Transformation 2 applied (Select). No execution yet.")

t3 = t2.withColumn("revenue", col("quantity") * col("unit_price"))
print("Transformation 3 applied (WithColumn). No execution yet.")

t4 = t3.groupBy("store_id").agg(sum("revenue").alias("store_revenue"))
print("Transformation 4 applied (GroupBy). No execution yet.")

t5 = t4.orderBy(col("store_revenue").desc())
print("Transformation 5 applied (OrderBy). No execution yet.")
t5.show()
print("Action triggered! Spark executed the entire DAG of transformations now.")

Transformation 1 applied (Filter). No execution yet.
Transformation 2 applied (Select). No execution yet.
Transformation 3 applied (WithColumn). No execution yet.
Transformation 4 applied (GroupBy). No execution yet.
Transformation 5 applied (OrderBy). No execution yet.
+--------+-------------+
|store_id|store_revenue|
+--------+-------------+
|     S07|    237480949|
|     S02|    236373766|
|     S03|    233939468|
|     S01|    231828344|
|     S08|    229851332|
|     S04|    220509065|
|     S06|    216437396|
|     S05|    214072770|
+--------+-------------+

Action triggered! Spark executed the entire DAG of transformations now.


In [ ]:
filtered_df = sales_df.filter(col("unit_price") > 15000)
filtered_df.explain(True)
print("Calling show():")
filtered_df.show(5)
print("Calling count():")
print("Count:", filtered_df.count())

== Parsed Logical Plan ==
'Filter '`>`('unit_price, 15000)
+- Relation [order_id#128,order_date#129,store_id#130,product_id#131,quantity#132,unit_price#133,discount_pct#134,payment_method#135,order_status#136,sales_channel#137] csv

== Analyzed Logical Plan ==
order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string
Filter (unit_price#133 > 15000)
+- Relation [order_id#128,order_date#129,store_id#130,product_id#131,quantity#132,unit_price#133,discount_pct#134,payment_method#135,order_status#136,sales_channel#137] csv

== Optimized Logical Plan ==
Filter (isnotnull(unit_price#133) AND (unit_price#133 > 15000))
+- Relation [order_id#128,order_date#129,store_id#130,product_id#131,quantity#132,unit_price#133,discount_pct#134,payment_method#135,order_status#136,sales_channel#137] csv

== Physical Plan ==
*(1) Filter (isnotnull(unit_price#133) AND (unit_pric

In [ ]:
grouped_df = sales_df.join(products_df, "product_id", "inner") \
                     .groupBy("category") \
                     .agg(sum("quantity").alias("total_quantity"))
results = grouped_df.collect()
for row in results:
    print(row)

Row(category='Computers', total_quantity=16258)
Row(category='Displays', total_quantity=7870)
Row(category='Mobiles', total_quantity=16572)
Row(category='Accessories', total_quantity=41839)
Row(category='Storage', total_quantity=17251)
Row(category='Audio', total_quantity=17159)
Row(category='Tablets', total_quantity=7915)


In [ ]:
result_df = sales_df.filter(col("order_status") == "Completed") \
                    .withColumn("discounted_revenue", (col("quantity") * col("unit_price")) * (1 - col("discount_pct") / 100))
result_df.write.mode("overwrite").parquet("completed_sales_output.parquet")
print("DataFrame successfully written to Parquet!")

DataFrame successfully written to Parquet!


In [ ]:
from google.colab import files
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()
# Browse and select multiple CSV files
uploaded = files.upload()
# Load selected CSV files into PySpark DataFrames
dataframes = {}
for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df
    print(f"\nFile: {filename}")
    print("Records:", df.count())
    df.show(5)

Saving lazy_execution_stores (1).csv to lazy_execution_stores (1) (3).csv

File: lazy_execution_stores (1) (3).csv
Records: 8
+--------+---------+------+-----------+
|store_id|     city|region| store_type|
+--------+---------+------+-----------+
|     S01|Hyderabad| South|       Mall|
|     S02|Bengaluru| South|High Street|
|     S03|  Chennai| South|       Mall|
|     S04|   Mumbai|  West|High Street|
|     S05|     Pune|  West|       Mall|
+--------+---------+------+-----------+
only showing top 5 rows


In [ ]:
sales_df = spark.read.csv("path/to/sales_data.csv", header=True, inferSchema=True)

In [ ]:
completed_df = sales_df.filter(F.col("order_status") == "Completed")

print("Transformation created")
print("Completed Orders:", completed_df.count())

In [ ]:
completed_df.cache()
print("Is Cached:", completed_df.is_cached)

completed_df.count()
completed_df.show(10)
completed_df.groupBy("product_id").count().show()

In [ ]:
high_quality_df = sales_df.filter(
    F.col("quantity") > 5
)

high_quality_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

high_quality_df.count()
high_quality_df.show(10)
print(high_quality_df.storageLevel)

NameError: name 'sales_df' is not defined

In [ ]:
from google.colab import files
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()
# Browse and select multiple CSV files
uploaded = files.upload()
# Load selected CSV files into PySpark DataFrames
dataframes = {}
for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df
    print(f"\nFile: {filename}")
    print("Records:", df.count())
    df.show(5)

In [23]:
import pyspark.sql.functions as F
sales_df = dataframes['lazy_execution_sales_25000 (1) (2).csv']

completed_df = sales_df.filter(F.col("order_status") == "Completed")

print("Transformation created successfully!")
print("Completed Orders:", completed_df.count())

Transformation created successfully!
Completed Orders: 21775


In [24]:
completed_df.cache()
print("Is Cached:", completed_df.is_cached)

completed_df.count()
completed_df.show(10)
completed_df.groupBy("product_id").count().show()

Is Cached: True
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       

In [27]:
from pyspark import StorageLevel
import pyspark.sql.functions as F
high_quality_df = sales_df.filter(
    F.col("quantity") > 5
)
high_quality_df.persist(
    StorageLevel.MEMORY_AND_DISK
)
high_quality_df.count()
high_quality_df.show(10)
print(high_quality_df.storageLevel)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|   

In [28]:
print("Total Records:", sales_df.count())
print("Number of Partitions:",
      sales_df.rdd.getNumPartitions())

Total Records: 25000
Number of Partitions: 1


In [32]:
partitioned_df = sales_df.repartition(8)

print("Original Partitions:", sales_df.rdd.getNumPartitions())
print("New Partitions:", partitioned_df.rdd.getNumPartitions())

partitioned_df.count()

Original Partitions: 1
New Partitions: 8


25000

In [ ]:
df8 = sales_df.repartition(8)

df_partition = df8.repartition(3)
df_coalesce = df8.coalesce(3)

print("Repartition:", df_partition.rdd.getNumPartitions())
print("Coalesce:", df_coalesce.rdd.getNumPartitions())

print("REPARTITION PLAN")
df_partition.explain()

print("COALESCE PLAN")
df_coalesce.explain()


In [39]:
print("Current number of partitions:", sales_df.rdd.getNumPartitions())

Current number of partitions: 1


In [40]:
df_12 = sales_df.repartition(12)
print("Increased partitions:", df_12.rdd.getNumPartitions())

Increased partitions: 12


In [41]:
print("Number of records in each partition:", sales_df.rdd.glom().map(len).collect())

Number of records in each partition: [25000]


In [42]:
df_product_6 = sales_df.repartition(6, "product_id")
print("Partitions using product_id:", df_product_6.rdd.getNumPartitions())

Partitions using product_id: 6


In [43]:
df_4 = df_12.coalesce(4)
print("Reduced partitions using coalesce:", df_4.rdd.getNumPartitions())

Reduced partitions using coalesce: 4
